# Lab 1 — Token Anatomy of Code

**Module:** Understanding Text Generation | **Duration:** 90 min | **Pair programming:** Switch roles at 40 min

**Fil Rouge:** DevAssist — AI-Powered Developer Documentation Assistant for TaskFlow

---

## Objectives

By the end of this lab, you will:
1. Distinguish tokens from words in both natural language and Python code
2. Predict the relative token cost of different prompt phrasings
3. Interpret temperature, top-k, and top-p effects empirically
4. Explain how prompt constraint interacts with sampling parameters

**Core principle:** *A prompt is not an incantation — it is a structured input that shapes which tokens the model considers probable.*

---
## Setup

Run the cells below to verify your environment. If anything fails, check the README or ask your instructor.

In [ ]:
# === Environment Check ===
import sys
print(f"Python: {sys.version}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import json
from collections import Counter

print("✓ All core imports successful!")

In [ ]:
# === Load Tokenizer ===
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B")
    print(f"✓ Loaded Llama 3.2 tokenizer (vocab size: {tokenizer.vocab_size})")
except Exception as e:
    print(f"⚠ Llama tokenizer unavailable ({e})")
    print("  Using GPT-2 fallback — token counts will differ but all concepts apply.")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    print(f"✓ Loaded GPT-2 tokenizer (vocab size: {tokenizer.vocab_size})")

In [ ]:
# === Check Ollama ===
OLLAMA_AVAILABLE = False
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✓ Ollama is running. Available models: {models}")
    OLLAMA_AVAILABLE = True
except Exception as e:
    print(f"⚠ Ollama not reachable: {e}")
    print("  Part A (Token Lab) works without Ollama.")
    print("  For Part B (Sampling Lab), you'll use pre-computed results.")

In [ ]:
# === Helper Functions ===

def analyze_tokens(text, label=""):
    """Tokenize text and display detailed results."""
    token_ids = tokenizer.encode(text)
    decoded = [tokenizer.decode([t]) for t in token_ids]
    char_count = len(text)
    token_count = len(token_ids)
    ratio = token_count / max(char_count, 1)
    print(f"\n{'=' * 60}")
    print(f"  Label:        {label}")
    print(f"  Input:        {text[:80]}{'...' if len(text) > 80 else ''}")
    print(f"  Token count:  {token_count}")
    print(f"  Char count:   {char_count}")
    print(f"  Tokens/char:  {ratio:.3f}")
    print(f"  Tokens:       {decoded[:12]}{'...' if len(decoded) > 12 else ''}")
    print(f"{'=' * 60}")
    return {
        "label": label, "text": text, "token_count": token_count,
        "char_count": char_count, "tokens_per_char": round(ratio, 3),
        "tokens": decoded, "ids": token_ids,
    }


def generate(prompt, temperature=1.0, top_k=40, top_p=0.9, n=1,
             model="llama3.2:3b", max_tokens=150):
    """Generate n completions using Ollama."""
    results = []
    for _ in range(n):
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={"model": model, "prompt": prompt, "stream": False,
                      "options": {"temperature": temperature, "top_k": top_k,
                                  "top_p": top_p, "num_predict": max_tokens}},
                timeout=60)
            results.append(response.json()["response"])
        except Exception as e:
            results.append(f"[Generation failed: {e}]")
    return results


# Fallback: load pre-computed outputs
def load_precomputed(section, key):
    """Load pre-computed outputs if Ollama is unavailable."""
    try:
        with open("data/precomputed_sampling.json", "r") as f:
            data = json.load(f)
        return data.get(section, {}).get(key, ["(no precomputed data)"])
    except FileNotFoundError:
        return ["(precomputed data file not found)"]


print("✓ Helper functions loaded.")

---
# PART A — TOKEN LAB (40 minutes)

**🟢 Driver starts here. Switch roles after Part A.**

---

## Exercise 1: Tokenize TaskFlow Code (15 min) — GUIDED

Run `analyze_tokens()` on the 20 samples below. These represent real content that DevAssist will encounter:

- **Group A:** Three paraphrases of the same PR summary instruction
- **Group B:** The same instruction in 4 languages
- **Group C:** TaskFlow code snippets (Python, JavaScript, SQL)
- **Group D:** Edge cases from real codebases
- **Group E:** Numbers and dev patterns
- **Group F:** Prompt engineering patterns

In [ ]:
# === TaskFlow-Relevant Text Samples ===

phrases = {
    # === Group A: Same instruction, different wording ===
    "en_formal":    "Please provide a comprehensive summary of this pull request, highlighting the key changes.",
    "en_casual":    "Sum up this PR for me — what changed?",
    "en_technical": "Generate an abstractive summary of the diff, listing modified functions and their purposes.",

    # === Group B: Same instruction in 4 languages ===
    "en_base": "Please summarize this pull request concisely.",
    "fr":      "Veuillez résumer cette pull request de manière concise.",
    "ar":      "يرجى تلخيص طلب السحب هذا بشكل موجز.",
    "zh":      "请简要总结这个拉取请求。",

    # === Group C: TaskFlow code snippets ===
    "python":     "def create_task(title: str, assignee: str, priority: int = 1) -> Task:",
    "javascript": "const fetchTasks = async (userId: string, filters: TaskFilter): Promise<Task[]> => {",
    "sql":        "SELECT t.title, u.name FROM tasks t JOIN users u ON t.assignee_id = u.id WHERE t.status = 'open';",

    # === Group D: Real codebase edge cases ===
    "emoji_commit": "feat: ✨ add user authentication with OAuth2 🔐 (#142)",
    "url":          "https://api.taskflow.dev/v2/tasks?status=open&assignee=alice&limit=50",
    "json_config":  '{"database": {"host": "localhost", "port": 5432, "name": "taskflow_db"}}',
    "markdown":     "## API Endpoints\n- `GET /tasks` — list all tasks\n- `POST /tasks` — create a new task\n```python\nresponse = client.get('/tasks')\n```",

    # === Group E: Numbers and dev patterns ===
    "metrics":   "Build #4523 passed: 98.7% coverage, 342 tests, avg latency 23ms, p99 = 187ms.",
    "iso_date":  "Sprint deadline: 2025-09-15T17:00:00Z. Retrospective: 2025-09-16T10:00:00+01:00.",
    "error_msg": "TaskNotFoundError: Task with id='task_a1b2c3d4' not found in database 'taskflow_prod'.",

    # === Group F: Prompt engineering patterns ===
    "system":   "You are DevAssist, a helpful coding assistant for the TaskFlow project. Respond only in JSON format.",
    "few_shot": "Input: 'fix login bug' → category: bug\nInput: 'add dark mode' → category: feature\nInput: 'update README' →",
    "cot":      "Let's think step by step. First, identify the function signature. Then, list the parameters. Finally, describe the return type.",
}

# Run tokenization on all 20 samples
results = {}
for key, phrase in phrases.items():
    results[key] = analyze_tokens(phrase, label=key)

---
## Exercise 2: Build Comparison Table (10 min) — SEMI-GUIDED

Create a sorted DataFrame from your results.

In [ ]:
# TODO: Build the comparison table

rows = []
for key, r in results.items():
    rows.append({
        "Label": r["label"],
        "Text (preview)": r["text"][:50] + ("..." if len(r["text"]) > 50 else ""),
        "Token Count": r["token_count"],
        "Tokens/Char": r["tokens_per_char"],
        # TODO: Add a column showing the first 8 decoded tokens
        "First 8 Tokens": str(r["tokens"][:8]),
    })

df = pd.DataFrame(rows)

# TODO: Sort by Token Count descending
df = df.sort_values("Token Count", ascending=False).reset_index(drop=True)

display(df)

### Analysis Questions

Answer each question in **3–5 sentences** in the cells below. Reference specific data from your table.

**Q1: Compare the three English paraphrases (Group A). Do they all cost the same number of tokens? What does this mean for prompt design in DevAssist?**

TODO

**Q2: Compare tokenization across languages (Group B). Which language is most "expensive"? Why?** *(Hint: think about the tokenizer's training data distribution.)*

TODO

**Q3: How does Python code tokenize (Group C)? Are variable names like `create_task` single tokens or split? What about operators like `->` and `:`?**

TODO

**Q4: Look at the few-shot prompt pattern (Group F). How do the separators (`→`, `\n`) tokenize? Could you save tokens by choosing a different separator?**

TODO

---
## Exercise 3: Paraphrase Sensitivity (15 min) — AUTONOMOUS

Write 4 paraphrases of the base prompt, then compare their token costs.

In [ ]:
base_prompt = "Extract all function names from the following Python code and return them as a JSON array."

paraphrases = [
    base_prompt,
    # TODO: Write 4 more paraphrases expressing the SAME intent.
    # Vary the style: formal, casual, code-like, minimal.
    # Example: "List every function defined in this Python code. Output: JSON array."
    "",  # v1: your paraphrase here
    "",  # v2: your paraphrase here
    "",  # v3: your paraphrase here
    "",  # v4: your paraphrase here
]

# Remove empty strings (in case student hasn't filled them all)
paraphrases = [p for p in paraphrases if p]

# Tokenize and compare
para_results = []
for i, p in enumerate(paraphrases):
    r = analyze_tokens(p, label=f"v{i}")
    para_results.append(r)

# Build comparison table
para_df = pd.DataFrame([{
    "Version": r["label"],
    "Prompt": r["text"][:60] + "...",
    "Tokens": r["token_count"],
    "Chars": r["char_count"],
    "Tokens/Char": r["tokens_per_char"],
} for r in para_results])

display(para_df.sort_values("Tokens"))

**Paraphrase Analysis:** Is the most token-efficient version also likely to be the most *effective* prompt? Why or why not? (5 sentences)

TODO

---
# PART B — SAMPLING LAB (40 minutes)

**🔄 Switch driver/navigator roles now.**

> **If Ollama is not available:** Use the pre-computed results. Replace `generate(...)` calls with `load_precomputed(section, key)` — see the fallback cells provided.

---

## Exercise 1: Temperature Sweep (20 min) — SEMI-GUIDED

Generate 10 completions at each temperature for the same TaskFlow-relevant prompt.

In [ ]:
prompt = "Explain in one paragraph why unit testing is important for a REST API like TaskFlow."

temperatures = [0.0, 0.2, 0.5, 0.7, 1.0, 1.5, 2.0]
sweep_results = {}

if OLLAMA_AVAILABLE:
    print("Running temperature sweep with Ollama... (this may take a few minutes)")
    for temp in temperatures:
        print(f"  Generating at temperature={temp}...")
        outputs = generate(prompt, temperature=temp, n=10)
        sweep_results[temp] = outputs
        print(f"    Sample: {outputs[0][:100]}...")
    print("\n✓ Sweep complete!")
else:
    print("Using pre-computed results (Ollama unavailable).")
    precomputed = json.load(open("data/precomputed_sampling.json"))
    for temp in temperatures:
        key = str(temp)
        sweep_results[temp] = precomputed.get("temperature_sweep", {}).get("results", {}).get(key, ["(no data)"])
    print("✓ Pre-computed results loaded.")

In [ ]:
# === TODO: Implement the three metrics ===
# After implementing, copy these functions to utils/sampling_utils.py for auto-grading.

from itertools import combinations


def measure_diversity(outputs):
    """Measure diversity across n outputs using unique bigrams.
    
    Returns: fraction of unique bigrams out of total bigrams (0.0 to 1.0).
    Higher = more diverse outputs.
    
    Algorithm:
        1. For each output, split into words (lowercase) and extract bigrams.
        2. Collect ALL bigrams from ALL outputs.
        3. Return len(unique_bigrams) / len(all_bigrams).
    """
    # TODO: Implement
    pass


def measure_stability(outputs):
    """Measure how similar outputs are using average pairwise Jaccard similarity.
    
    Returns: average Jaccard similarity (0.0 to 1.0).
    Higher = more similar/stable outputs.
    
    Algorithm:
        1. For each output, create a set of lowercase words.
        2. For each pair (i,j), compute Jaccard: |A ∩ B| / |A ∪ B|.
        3. Return average across all pairs.
    """
    # TODO: Implement
    pass


def measure_coherence(outputs):
    """Basic coherence: fraction of well-formed outputs.
    
    Well-formed = ends with punctuation AND no excessive repetition.
    
    Returns: fraction of well-formed outputs (0.0 to 1.0).
    
    Algorithm:
        1. Check if output ends with . ! ? \" )
        2. Check if any 3-word sequence appears 3+ times.
        3. Well-formed = ends_ok AND NOT has_repetition.
    """
    # TODO: Implement
    pass

In [ ]:
# === Apply metrics and build summary table ===

summary_rows = []
for temp, outputs in sweep_results.items():
    row = {
        "Temperature": temp,
        "Avg Length (chars)": int(np.mean([len(o) for o in outputs])),
    }
    # Add metrics if implemented
    try:
        row["Diversity"] = round(measure_diversity(outputs), 3)
    except (TypeError, NotImplementedError):
        row["Diversity"] = "(not implemented)"
    try:
        row["Stability"] = round(measure_stability(outputs), 3)
    except (TypeError, NotImplementedError):
        row["Stability"] = "(not implemented)"
    try:
        row["Coherence"] = round(measure_coherence(outputs), 3)
    except (TypeError, NotImplementedError):
        row["Coherence"] = "(not implemented)"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# === TODO: Create a chart ===
# Plot at least one metric as a function of Temperature.
# Bonus: plot all three metrics with different colors.

fig, ax = plt.subplots(figsize=(10, 6))

# TODO: Plot your metrics here
# Example (uncomment and adapt once metrics are implemented):
#
# numeric_df = summary_df[summary_df["Diversity"] != "(not implemented)"].copy()
# ax.plot(numeric_df["Temperature"], numeric_df["Diversity"], 'o-', label="Diversity", color="#2196F3")
# ax.plot(numeric_df["Temperature"], numeric_df["Stability"], 's-', label="Stability", color="#4CAF50")
# ax.plot(numeric_df["Temperature"], numeric_df["Coherence"], '^-', label="Coherence", color="#FF9800")

ax.set_xlabel("Temperature", fontsize=12)
ax.set_ylabel("Metric Value (0–1)", fontsize=12)
ax.set_title("Effect of Temperature on Generation Quality — TaskFlow Unit Testing Prompt", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

### Temperature Sweep — Guiding Questions

**Q1: At which temperature do you start seeing incoherent or nonsensical text?**

TODO

**Q2: At temperature 0.0, are all 10 outputs identical? If not, why not?**

TODO

**Q3: For a factual task like this, what temperature would you recommend? Why?**

TODO

---
## Exercise 2: Top-p Comparison (10 min) — SEMI-GUIDED

In [ ]:
creative_prompt = "Write a creative commit message for a feature that adds user authentication to TaskFlow."

top_p_values = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0]
topp_results = {}

if OLLAMA_AVAILABLE:
    for tp in top_p_values:
        outputs = generate(creative_prompt, temperature=0.7, top_p=tp, n=5)
        topp_results[tp] = outputs
        print(f"\ntop_p={tp}:")
        for out in outputs:
            print(f"  → {out.strip()[:100]}")
else:
    print("⚠ Ollama unavailable. Analyze the temperature sweep results above instead.")
    print("  Note in your analysis that you would expect similar diversity/quality patterns with top-p.")

**Top-p Analysis:** At what top-p value do you see the biggest jump in creativity/diversity? At what point does quality degrade?

TODO

---
## Exercise 3: Prompt Constraint × Sampling Interaction (10 min) — AUTONOMOUS

**Hypothesis:** A tightly constrained prompt is LESS sensitive to temperature changes.

In [ ]:
constraint_prompts = {
    "loose": "Tell me about TaskFlow.",
    
    "medium": "Explain 3 advantages of using FastAPI for TaskFlow's REST API in bullet points.",
    
    "tight": """Extract the function name and parameters from this code.
Return ONLY a JSON object with keys "name" and "params" (list of strings).
No explanation.

Code: def create_task(title: str, assignee: str, priority: int = 1) -> Task:"""
}

test_temps = [0.0, 0.7, 1.5]

if OLLAMA_AVAILABLE:
    for label, prompt in constraint_prompts.items():
        print(f"\n{'=' * 60}")
        print(f"Prompt type: {label.upper()}")
        print(f"Prompt: {prompt[:80]}...")
        for temp in test_temps:
            outputs = generate(prompt, temperature=temp, n=3)
            print(f"\n  temp={temp}:")
            for out in outputs:
                print(f"    → {out.strip()[:120]}")
else:
    print("Using pre-computed results:")
    precomputed = json.load(open("data/precomputed_sampling.json"))
    for label in ["loose", "medium", "tight"]:
        print(f"\n{'=' * 60}")
        print(f"Prompt type: {label.upper()}")
        for temp in ["0.0", "0.7", "1.5"]:
            outputs = precomputed.get("constraint_interaction", {}).get(label, {}).get(temp, ["(no data)"])
            print(f"\n  temp={temp}:")
            for out in outputs:
                print(f"    → {out[:120]}")

**Constraint × Sampling Analysis (5 sentences):**

For each prompt type, assess how much the output varies across temperatures. Does a tighter prompt reduce sensitivity to temperature? Why does this make sense mechanistically? *(Use the terms: token distribution, probability, logits, context.)*

TODO

---
# PORTFOLIO ENTRY — Use Case #1: Code Completion (v0)

This is the **first entry** in your prompt portfolio. You will iterate on it (v1, v2) in later labs.

**Task:** Write a v0 prompt for asking DevAssist to complete a Python function in the TaskFlow codebase.

In [ ]:
# === Portfolio Entry v0: Code Completion ===

# TODO: Write your v0 prompt below.
# It should ask the model to complete the get_overdue_tasks function.
portfolio_prompt = """
Complete the following Python function. Return ONLY the function body.

def get_overdue_tasks(tasks: list[Task], current_date: datetime) -> list[Task]:
    \"\"\"Return all tasks whose due_date is before current_date and status is not 'completed'.\"\"\"
"""

# Token count of your prompt
prompt_tokens = tokenizer.encode(portfolio_prompt)
print(f"Prompt token count: {len(prompt_tokens)}")
print(f"Prompt tokens: {[tokenizer.decode([t]) for t in prompt_tokens[:20]]}...")

if OLLAMA_AVAILABLE:
    print("\n--- Temperature 0.2 ---")
    out_low = generate(portfolio_prompt, temperature=0.2, n=1)
    print(out_low[0])

    print("\n--- Temperature 0.8 ---")
    out_high = generate(portfolio_prompt, temperature=0.8, n=1)
    print(out_high[0])
else:
    print("\n⚠ Ollama unavailable. Write your observations based on the sampling lab results.")
    print("At T=0.2, you would expect: a deterministic, clean list comprehension.")
    print("At T=0.8, you would expect: potentially different implementation styles (filter, loop, etc.)")

### Portfolio v0 — Mechanistic Observation (3 sentences)

Compare the two outputs (or what you would expect). What differences do you observe? Explain them in terms of token distributions and sampling.

TODO

---
# WRAP-UP

## Key Takeaways

Write 3 bullet points summarizing what you learned today:

1. TODO
2. TODO
3. TODO

---

## Before You Submit

1. ✅ All code cells executed (no errors)
2. ✅ All TODO markers replaced with your answers
3. ✅ Comparison table complete (20 samples)
4. ✅ At least one chart generated
5. ✅ Portfolio entry v0 written
6. ✅ Copy your `measure_diversity`, `measure_stability`, `measure_coherence` functions to `utils/sampling_utils.py`
7. ✅ `git add -A && git commit -m 'Lab 1 complete' && git push`

---

## Homework: Tokenization Journal

**Due before Module 2.** Create `tokenization_journal.md` (1 page):
- Choose one DevAssist task (e.g., "extract TODO comments from Python code")
- Write the prompt in 3+ formulations (formal, casual, another language/code-like)
- Show tokenization for each
- Discuss implications for token efficiency, multilingual deployment, reliability
- Reference BPE/SentencePiece mechanisms

---

## Preview: Lab 2

Today you learned **WHAT** the model sees (tokens) and **HOW** it samples (temperature, top-k, top-p). In Lab 2, you'll discover **WHY** context placement matters — because the **attention mechanism** decides which tokens influence the output most.

---

*Lab 1 of 8 — DevAssist / TaskFlow Lab Series*